<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="./images/btp-banner.gif" alt="BTP A&C">
</div>

## Retrieval-Augmented Generation with SAP AI Core Document Grounding

In this demo, we will explore how to enhance the capabilities of Large Language Models (LLMs) with **SAP AI Core Document Grounding**. You will learn how to index unstructured and semi-structured data using AI models from **SAP Generative AI Hub**, and store the document chunks in **SAP AI Core Document Grounding**. Additionally, you will retrieve relevant document chunks using semantic search, and forward the relevant results to a LLM to generate an augmented answer.

## 🎯 Learning Objectives
By the end of this demo, you will be able to:
- Implement a full RAG pipeline using Python, LangChain, and Generative AI Hub SDK.
- Index document chunks into SAP AI Core Document Grounding collections.
- Retrieve the most relevant content based on semantic similarity using Document Grounding retrieval.
- Augment user prompts with retrieved context and invoke LLMs to generate more accurate, grounded answers.
- Design and use prompt templates to enhance the quality of generated responses.

## 🚨 Requirements

Before starting the Jupyter Notebook steps, ensure the following:
- SAP AI Core instance with **Document Grounding** capability enabled
- Deploy AI models in SAP AI Launchpad:
  - Large Language Model (LLM) for chat/completion: **`anthropic--claude-4.5-sonnet`**
  - Embedding model for vector representations: **`text-embedding-3-large`**

## 📝 About the Data

The data set is a product catalog of IT accessory products. Here are the main attributes and their descriptions based on the sample data:

|Field          |Description            |
----------------|-----------------------
|**PRODUCT_ID**| A unique identifier for each product.|
|**PRODUCT_NAME**| The name of the product, which typically includes the brand and the model.|
|**CATEGORY**| The general category of the product, which is "IT Accessories" for all entries sampled.|
|**DESCRIPTION**| A detailed description of the product, highlighting key features and specifications.|
|**UNIT_PRICE**| The price of the product in Euros.|
|**UNIT_MEASURE**| The unit of measure for the product, typically "Each" indicating pricing per item.|
|**SUPPLIER_ID**| A unique identifier for the supplier of the product.|
|**SUPPLIER_NAME**| The name of the supplier.|
|**LEAD_TIME_DAYS**| The number of days it takes from order to delivery.|
|**MIN_ORDER**| The minimum order quantity required.|
|**CURRENCY**| The currency of the transaction, which is "EURO" for all entries.|
|**SUPPLIER_COUNTRY**| The country where the supplier is located, which is "Germany" for all sampled entries.|
|**SUPPLIER_ADDRESS**| The physical address of the supplier.|
|**AVAILABILITY_DAYS**| The number of days the product is available for delivery.|
|**SUPPLIER_CITY**| The city where the supplier is located.|
|**STOCK_QUANTITY**| The quantity of the product currently in stock.|
|**MANUFACTURER**| The company that manufactured the product.|
|**CITY_LAT**| Geographical coordinates of the city (latitude)|
|**CITY_LONG**| Geographical coordinates of the city (longitude).|
|**RATING**| A rating for the product, which are on a scale from 1 to 5.|

</br>

This dataset is structured to support various business operations such as inventory management, order processing, and logistics planning, providing a comprehensive view of product offerings, supplier details, and stock levels.

</br>


## Retrieval Augmented Generation using Generative AI Hub and SAP AI Core Document Grounding

### Hands-on Retrieval Augmented Generation (RAG) workflow

The Retrieval Augmented Generation use case process consists of steps to be completed as seen in the graphic below.

<br>

> ![title](./images/rag_full.png)

<br>

#### Indexing Process
1. Business documents that should be used for answering user questions are fed into the model. The contents of the files are split into smaller chunks.
    >"Chunking" refers to dividing a large text corpus into smaller, manageable pieces or segments. Each chunk acts as a standalone unit of information that can be individually indexed and retrieved.
2. Embedding functions are used to create embeddings from the file/document chunks.
    >Embeddings refer to dense, continuous vectors representing text in a high-dimensional space. These vectors serve as coordinates in a semantic space, capturing the relationships and meanings between words.
3. The embeddings are then stored as vectors in the **SAP AI Core Document Grounding** service.

#### Retrieval Process
1. A query or prompt is submitted.
2. The query is embedded into a vector form.
3. The query vector is compared to the values stored in SAP AI Core Document Grounding via a similarity/semantic search.
4. The most appropriate and relevant results are identified.
5. And forwarded, along with the original query, to a large language model.
6. The LLM uses the results of the Document Grounding search to augment its own searching capabilities, and the final answer is returned to the user.

### Implementing RAG with Document Grounding

- Prepare the documentation for the product catalog in CSV format with each row representing a product.
- Connect to the SAP AI Core Document Grounding service and create a **collection** to store the documentation data.
- Populate the collection with document chunks using the Document Grounding REST API.
- Use the Document Grounding retrieval API to perform semantic search and retrieve relevant results.

### Enhancing Query Responses

- Define a prompt template to provide context to queries.
- Modify the function to query the LLM (Large Language Model) based on the prompt template.
- Test the model's response using specific queries related to the product catalog, ensuring it provides contextually relevant responses based on retrieved document chunks.

> Retrieval augmented generation optimizes the output of large language models by applying more context to prompts.

## Setup and configuration

The following Python modules are to be installed during this hands-on introduction.

#### **sap-ai-sdk-gen**

With this SAP python SDK you can leverage the power of generative Models like chatGPT available in SAP's generative AI Hub.

For more information, please see https://pypi.org/project/sap-ai-sdk-gen/

#### **requests**

The `requests` library is used to make HTTP calls to the SAP AI Core Document Grounding REST API for creating collections, uploading documents, and performing semantic retrieval.

For more information, please see https://pypi.org/project/requests/

</br>

#### Install Python packages

Run the following package installations. **pip** is the package installer for Python. You can use pip to install packages from the Python Package Index and other indexes.

In [ ]:
%pip install "sap-ai-sdk-gen[all]" --break-system-packages -U
%pip install requests --break-system-packages -U

# kernel restart required!!!

#### Restart Python kernel

The Python kernel needs to be restarted before continuing.

> ![title](./images/config_001.png)

#### Configure SAP Generative AI Hub credentials

Execute the configuration module below to enable access to SAP Generative AI foundation models. This configuration is automatically done by running configuration module in the code block.

You could also set up the same by running a terminal command: **aicore configure**

</br>

> Please ensure that the Python kernel was restarted!


In [ ]:
# Generative AI Config
import os
import json

with open(os.path.join(os.getcwd(), 'env_config.json')) as f:
    aicore_config = json.load(f)

### Initialize the LLM model
LLM is initialized with a model **anthropic--claude-4.5-sonnet**. This is used for generating responses or interacting in a chat-like environment.

> **IMPORTANT!** here you are connecting to the **anthropic--claude-4.5-sonnet** model that was deployed in SAP AI Core.

In [ ]:
# Set llm
from ai_core_sdk.ai_core_v2_client import AICoreV2Client
from gen_ai_hub.proxy.gen_ai_hub_proxy import GenAIHubProxyClient
# harmonized init helper
from gen_ai_hub.proxy.langchain.init_models import init_llm

# Set up the AICoreV2Client
ai_core_client = AICoreV2Client(base_url=aicore_config['AICORE_BASE_URL'],
                            auth_url=aicore_config['AICORE_AUTH_URL'],
                            client_id=aicore_config['AICORE_CLIENT_ID'],
                            client_secret=aicore_config['AICORE_CLIENT_SECRET'],
                            resource_group=aicore_config['AICORE_RESOURCE_GROUP'])
# Set up the GenAIHubProxyClient
proxy_client = GenAIHubProxyClient(ai_core_client=ai_core_client)
print("\u2705 AI Core Client connection is established successfully!")

# Set up the LLM model
llm = init_llm('anthropic--claude-4.5-sonnet', proxy_client=proxy_client, max_tokens=800, temperature=0.3, top_p=None)
print("\u2705 LLM model connection is established successfully!")

### Ask LLM without context

After completing the configuration we are ready to ask the first question directly to LLM (anthropic--claude-4.5-sonnet) without any business product context to find us products with a rating of 4 and more. The response is arbitrary and does not relate to our product data.

</br>

> **Note** We can solve this problem by following the next steps in implementing RAG with Document Grounding.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from IPython.display import Markdown

# Define a simple prompt template for querying the LLM to get a keyboard suggestion based on the information from the Internet (for now).
TEMPLATE = """Advise the user based on the information available to you from general sources. Question: {question}"""

# Create a HumanMessagePromptTemplate using the defined template.
human_message_prompt = HumanMessagePromptTemplate.from_template(TEMPLATE)
# Create a ChatPromptTemplate using the human message prompt defined above.
chat_prompt = ChatPromptTemplate.from_messages([human_message_prompt])

question = "Return one keyboard suggestion (brand + model) with rating greater than 4. Format as: 'Keyboard: <name>'"

# Format the prompt with the input question
prompt_text = chat_prompt.format_prompt(question=question).to_string()

# Invoke the LLM with the formatted prompt and get the response
llm_response = llm.invoke(prompt_text)

# Display the LLM response in a readable format
display(Markdown(llm_response.content.strip()))

# Implementing Retrieval Augmented Generation (RAG)

### Prepare the documentation for the product catalog in CSV format with each row representing a product

This code snippet demonstrates how to load and process text data from a CSV file using the `CSVLoader` from the `langchain_community.document_loaders` library.

This process is useful for handling large text data, making it more manageable or suitable for further processing, analysis, or input into machine learning models, especially when dealing with limitations on input size.

In [ ]:
from langchain_community.document_loaders import CSVLoader

FILE_PRODUCTS = "data/product_catalog.csv"

# Process CSV data file
loader = CSVLoader(
    file_path=FILE_PRODUCTS,
    source_column="PRODUCT_ID",
    metadata_columns=[
        "SUPPLIER_ID",
        "CATEGORY",
        "SUPPLIER_COUNTRY",
        "SUPPLIER_CITY",
        "MANUFACTURER",
    ],
    content_columns=[
        "PRODUCT_NAME",
        "DESCRIPTION",
        "UNIT_PRICE",
        "LEAD_TIME_DAYS",
        "STOCK_QUANTITY",
        "RATING",
        "MIN_ORDER",
        "CATEGORY",
        "SUPPLIER_NAME",
        "SUPPLIER_COUNTRY",
        "SUPPLIER_CITY",
        "SUPPLIER_ADDRESS",
        "STATUS",
        "CURRENCY",
        "MANUFACTURER",
        "CITY_LAT",
        "CITY_LONG",
    ],
    csv_args={"delimiter": ";", "quotechar": '"'},
    encoding="utf-8-sig",
)

# Process data
text_documents = loader.load()
print(f"Loaded {len(text_documents)} documents")

for chunks in text_documents:
    print(chunks.metadata)
    print(chunks.page_content)

> At this point we have implemented the first RAG step - generated text chunks from source data
> 
> ![rag_indexing_1](./images/rag_indexing_1.png)

### SAP AI Core Document Grounding

[The **grounding module** in SAP AI Core](https://help.sap.com/docs/sap-ai-core/generative-ai/grounding-035c455a5a424697b60f4a24b6d791fe) integrates external, domain-specific, or real-time data into generative AI workflows. By enriching pretrained models with contextually relevant information, grounding improves response accuracy and relevance beyond the models' general training data.

Pretrained generative models are trained on broad, general-purpose datasets and do not have direct access to your organization's proprietary or real-time data. Grounding addresses this limitation by:
- Connecting models to external data sources
- Retrieving relevant information at runtime
- Supplying that information as context for generation

#### Two Approaches to Prepare the Knowledge Base

SAP AI Core Document Grounding offers two options for providing data:

| | Option 1: Pipelines API | Option 2: Vector API (this notebook) |
|---|---|---|
| **How it works** | Upload documents to a supported data repository (SharePoint, S3, SFTP, SAP DMS, etc.) and run a data pipeline that automatically chunks, embeds, and indexes the documents | Provide document chunks directly via the Vector API — you control chunking and content |
| **Best for** | Large document repositories with scheduled refresh (daily sync) | Structured data, custom chunking logic, or programmatic ingestion |
| **Supported sources** | Microsoft SharePoint, AWS S3, SFTP, SAP Build Work Zone, SAP Document Management, ServiceNow, Google Drive | Any data you can load in Python |
| **API** | `POST /pipelines` | `POST /vector/collections` + `POST /vector/collections/{id}/documents` |

<br/>

> **This notebook uses Option 2 (Vector API directly).** We load the product catalog CSV in Python, convert each row into a document chunk, and upload it to a Document Grounding collection using the Vector API. This gives us full control over the content and metadata of each chunk.

#### Grounding Architecture

![image.png](images/ai_core_grounding_architecture.png)

</br>

#### Connect to the SAP AI Core Document Grounding service

The following code establishes a connection to the Document Grounding REST API using OAuth2 authentication. We obtain an access token from the AI Core authorization server and set up the HTTP headers required for all subsequent API calls.

In [ ]:
import requests
import time

# Obtain an OAuth2 access token for the AI Core Document Grounding API
def get_access_token(auth_url, client_id, client_secret):
    """Retrieve an OAuth2 client credentials access token."""
    response = requests.post(
        f"{auth_url}/oauth/token",
        data={
            "grant_type": "client_credentials",
            "client_id": client_id,
            "client_secret": client_secret,
        },
    )
    response.raise_for_status()
    return response.json()["access_token"]

# Get the access token
access_token = get_access_token(
    aicore_config['AICORE_AUTH_URL'],
    aicore_config['AICORE_CLIENT_ID'],
    aicore_config['AICORE_CLIENT_SECRET'],
)

# Set up common HTTP headers for all Document Grounding API calls
dg_headers = {
    "Authorization": f"Bearer {access_token}",
    "AI-Resource-Group": aicore_config['AICORE_RESOURCE_GROUP'],
    "Content-Type": "application/json",
}

# Base URL for the Document Grounding API
# Vector operations use: {dg_base_url}/vector/collections
# Search uses:           {dg_base_url}/vector/search
dg_base_url = f"{aicore_config['AICORE_BASE_URL']}/lm/document-grounding"

print("\u2705 Document Grounding connection established successfully!")
print(f"   API endpoint: {dg_base_url}")

## Create a Document Grounding Collection and Store Embeddings

We will create a **collection** in SAP AI Core Document Grounding to store our product catalog documents. A collection is a logical container that groups related documents and their vector embeddings together.

The collection is configured with an embedding model (`text-embedding-3-large`) that will be used to automatically generate vector representations of the uploaded document chunks.

### Create a collection in Document Grounding

A collection in Document Grounding is equivalent to a table in a vector database. It stores document chunks along with their vector embeddings and metadata, enabling fast semantic search.

> **Note:** Collection creation is **asynchronous**. The API returns `202 Accepted` with a `Location` header pointing to a status URL. We poll that URL until the collection status is `created`.

In [ ]:
# Create a new Document Grounding collection for the product catalog
# POST /vector/collections returns 202 Accepted (async operation)
collection_title = "products-it-accessories"

create_response = requests.post(
    f"{dg_base_url}/vector/collections",
    headers=dg_headers,
    json={
        "title": collection_title,
        "embeddingConfig": {
            "modelName": "text-embedding-3-large"
        }
    },
)
create_response.raise_for_status()  # Expects 202 Accepted

# Extract the collection ID from the Location header
# Location format: .../vector/collections/{id}/creationStatus
location = create_response.headers.get("Location", "")
collection_id = location.split("/collections/")[-1].split("/")[0]

# Build the full status URL
if location.startswith("http"):
    status_url = location
else:
    status_url = f"{aicore_config['AICORE_BASE_URL']}{location}"

# Poll until the collection is ready
print(f"   Waiting for collection '{collection_title}' to be ready...")
for attempt in range(30):
    status_resp = requests.get(status_url, headers=dg_headers)

    # 204 No Content or empty body means the collection is ready
    if status_resp.status_code == 204 or not status_resp.text.strip():
        print(f"\u2705 Collection created successfully!")
        print(f"   Collection ID   : {collection_id}")
        print(f"   Collection title: {collection_title}")
        break

    try:
        status_data = status_resp.json()
        current_status = status_data.get("status", "pending")
        if current_status == "created":
            print(f"\u2705 Collection created successfully!")
            print(f"   Collection ID   : {collection_id}")
            print(f"   Collection title: {collection_title}")
            break
        print(f"   Attempt {attempt + 1}: status={current_status}, retrying...")
    except Exception:
        print(f"   Attempt {attempt + 1}: unexpected response (status={status_resp.status_code}), retrying...")

    time.sleep(2)
else:
    raise TimeoutError("Collection creation timed out after 60 seconds")

### Upload product documents to the collection

We convert the loaded CSV documents into the Document Grounding format and upload them to the collection. Each document consists of:
- **metadata**: key-value pairs where each value is a **list of strings** (required by the API schema)
- **chunks**: the actual text content that will be embedded and indexed for retrieval

In [ ]:
# Convert LangChain documents to Document Grounding format and upload them
# Documents are uploaded in batches to avoid request size limits

BATCH_SIZE = 10  # Number of documents per upload batch

def upload_documents_batch(documents_batch):
    """Upload a batch of documents to the Document Grounding collection."""
    dg_documents = []
    for doc in documents_batch:
        # Convert metadata dict to list of key-value pairs
        # NOTE: 'value' must be a list of strings per the VectorDocumentKeyValueListPair schema
        doc_metadata = [
            {"key": k, "value": [str(v)]}
            for k, v in doc.metadata.items()
            if v is not None
        ]
        dg_documents.append({
            "metadata": doc_metadata,
            "chunks": [
                {
                    "content": doc.page_content,
                    "metadata": []
                }
            ]
        })

    upload_response = requests.post(
        f"{dg_base_url}/vector/collections/{collection_id}/documents",
        headers=dg_headers,
        json={"documents": dg_documents},
    )
    upload_response.raise_for_status()
    return upload_response.json()

# Upload all documents in batches
total_uploaded = 0
for i in range(0, len(text_documents), BATCH_SIZE):
    batch = text_documents[i : i + BATCH_SIZE]
    upload_documents_batch(batch)
    total_uploaded += len(batch)
    print(f"   Uploaded batch {i // BATCH_SIZE + 1}: {total_uploaded}/{len(text_documents)} documents")

print(f"\u2705 All {total_uploaded} product documents uploaded to collection '{collection_title}'!")

### Verify documents stored in the collection

After uploading, we verify that the documents have been successfully stored in the Document Grounding collection by listing the documents and inspecting their metadata.

In [ ]:
# Retrieve and display the documents stored in the collection
# GET /vector/collections/{collectionId}/documents returns DocumentWithoutChunks objects
list_response = requests.get(
    f"{dg_base_url}/vector/collections/{collection_id}/documents",
    headers=dg_headers,
    params={"$top": 5},
)
list_response.raise_for_status()
stored_docs = list_response.json()

print(f"\u2705 Documents in collection '{collection_title}':")
print(f"   Total documents: {stored_docs.get('count', 'N/A')}")
print()

# Display a sample of stored documents (metadata only - chunks not returned by list endpoint)
for i, doc in enumerate(stored_docs.get("resources", [])[:3]):
    print(f"--- Document {i + 1} ---")
    print(f"ID: {doc.get('id', 'N/A')}")
    for m in doc.get("metadata", [])[:3]:
        print(f"  {m.get('key')}: {m.get('value')}")
    print()

> At this point we have implemented the Indexing Process part of RAG
> 
> ![rag_indexing_2](./images/rag_indexing_2.png)

## Performing a Semantic Search to Find Relevant Products

## Vector Search Using SAP AI Core Document Grounding

In this step, we use the **`/vector/search`** endpoint to convert a natural language query into a vector representation and retrieve the most relevant product records from the collection using **semantic similarity search**.

The search request (`TextSearchRequest`) accepts:
- A natural language **`query`** string
- A list of **`filters`** (`VectorSearchFilter`) specifying which collections to search and how many results to return

The service automatically embeds the query and returns the most semantically similar document chunks.

In [ ]:
# Perform semantic search using POST /vector/search
# TextSearchRequest: { query, filters: [VectorSearchFilter] }
# VectorSearchFilter: { id (unique per request), collectionIds, configuration: { maxChunkCount } }

def search_documents(query, collection_id, top_k=4):
    """Search for relevant documents using semantic similarity via /vector/search."""
    search_response = requests.post(
        f"{dg_base_url}/vector/search",
        headers=dg_headers,
        json={
            "query": query,
            "filters": [
                {
                    "id": "filter-1",
                    "collectionIds": [collection_id],
                    "configuration": {"maxChunkCount": top_k}
                }
            ]
        },
    )
    search_response.raise_for_status()
    return search_response.json()

def extract_chunks(search_results):
    """Extract chunk content strings from VectorSearchResults.
    Response structure:
      results -> VectorPerFilterSearchResult
        -> results -> DocumentsChunk
          -> documents -> Document-Output
            -> chunks -> VectorChunk { id, content, metadata }
    """
    chunks = []
    for filter_result in search_results.get("results", []):
        for collection_result in filter_result.get("results", []):
            for doc in collection_result.get("documents", []):
                for chunk in doc.get("chunks", []):
                    chunks.append(chunk.get("content", ""))
    return chunks

# Test the semantic search
search_query = "keyboard with rating 4 or higher"
search_results = search_documents(search_query, collection_id, top_k=4)
chunks = extract_chunks(search_results)

print(f"Search query: '{search_query}'")
print(f"Retrieved {len(chunks)} chunk(s)")
print()

for i, content in enumerate(chunks):
    print(f"--- Chunk {i + 1} ---")
    print(content)
    print()

> At this point we have implemented the Retrieval Process part of RAG
> 
> ![rag_retrieval_1](./images/rag_retrieval_1.png)

## Augmenting the LLM with Retrieved Context

Now we combine the retrieved document chunks with the user's question and pass them to the LLM. This is the **Augmented Generation** step of RAG.

We define a prompt template that instructs the LLM to:
1. Use **only** the provided context to answer the question
2. Return structured, relevant results from the product catalog
3. Avoid hallucinating information not present in the retrieved chunks

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from IPython.display import Markdown

# Define a RAG prompt template that instructs the LLM to use the retrieved context
RAG_TEMPLATE = """You are a helpful product advisor. Use ONLY the following product catalog context to answer the question.
If the answer is not found in the context, say 'I could not find relevant products in the catalog.'

Context:
{context}

Question: {question}

Answer based on the catalog data above:"""

human_message_prompt = HumanMessagePromptTemplate.from_template(RAG_TEMPLATE)
rag_chat_prompt = ChatPromptTemplate.from_messages([human_message_prompt])

def rag_query(question, collection_id, top_k=4):
    """Perform a full RAG query: retrieve relevant chunks and augment the LLM response."""
    # Step 1: Retrieve relevant document chunks from Document Grounding
    search_results = search_documents(question, collection_id, top_k=top_k)

    # Step 2: Extract chunk content and build context string
    context_parts = extract_chunks(search_results)
    context = "\n\n".join(context_parts)

    # Step 3: Format the prompt with the retrieved context and the user question
    prompt_text = rag_chat_prompt.format_prompt(context=context, question=question).to_string()

    # Step 4: Invoke the LLM with the augmented prompt
    llm_response = llm.invoke(prompt_text)
    return llm_response.content.strip(), context_parts

print("\u2705 RAG query function defined successfully!")

### Test the RAG pipeline with product catalog queries

Now we test the complete RAG pipeline with questions about our product catalog. The LLM will use the retrieved product data to provide accurate, grounded answers.

In [ ]:
# Test Query 1: Find keyboards with high ratings
question1 = "Return one keyboard suggestion (brand + model) with rating >=4. Format as: 'Keyboard: <name>'"
print(f"Question: {question1}")
print()

answer1, context1 = rag_query(question1, collection_id, top_k=4)

print("Retrieved context chunks:")
for i, chunk in enumerate(context1):
    print(f"  Chunk {i+1}: {chunk[:150]}...")
print()
print("LLM Answer (with RAG context):")
display(Markdown(answer1))

In [ ]:
# Test Query 2: Find products from a specific supplier city
question2 = "Which products are available from suppliers in Berlin? List the product names and prices."
print(f"Question: {question2}")
print()

answer2, context2 = rag_query(question2, collection_id, top_k=5)

print("LLM Answer (with RAG context):")
display(Markdown(answer2))

In [ ]:
# Test Query 3: Find products under a price threshold
question3 = "What are the cheapest IT accessories available under 50 EUR? List up to 3 products with their prices."
print(f"Question: {question3}")
print()

answer3, context3 = rag_query(question3, collection_id, top_k=5)

print("LLM Answer (with RAG context):")
display(Markdown(answer3))

## Summary

In this notebook, we successfully implemented a complete **Retrieval-Augmented Generation (RAG)** pipeline using **SAP AI Core Document Grounding**:

1. **Data Loading**: Loaded the product catalog from a CSV file using LangChain's `CSVLoader`.
2. **Connection**: Established a connection to the SAP AI Core Document Grounding REST API using OAuth2 authentication.
3. **Indexing**: Created a collection (`POST /vector/collections`) and uploaded all product documents (`POST /vector/collections/{id}/documents`).
4. **Retrieval**: Used `POST /vector/search` with a `VectorSearchFilter` to find relevant product chunks for user queries.
5. **Augmented Generation**: Combined retrieved context with user questions and invoked the LLM to generate accurate, grounded answers.

### Key Benefits of SAP AI Core Document Grounding
- **Fully managed**: No need to deploy or maintain a separate vector database
- **Integrated**: Seamlessly works with SAP Generative AI Hub models
- **Scalable**: Handles enterprise-scale document collections
- **Secure**: Built-in OAuth2 authentication and resource group isolation